## 使用模块与 VPRTempo 和 VPRTempoQuant

### 作者 Adam D Hines (https://research.qut.edu.au/qcr/people/adam-hines/)

VPRTempo 基于以下论文，如果您使用或发现此代码对您的研究有帮助，请考虑引用来源：

[Adam D Hines, Peter G Stratton, Michael Milford, & Tobias Fischer. "VPRTempo: A Fast Temporally Encoded Spiking Neural Network for Visual Place Recognition. arXiv September 2023](https://arxiv.org/abs/2309.10225)

### 介绍

在本教程中，我们将介绍如何在 VPRTempo 中使用模块。模块将训练数据分解为多个网络，这已被证明可以[提高整体性能](https://towardsdatascience.com/machine-learning-with-expert-models-a-primer-6c74585f223f)和较大模型的准确性。

开始之前，请确保您已[安装依赖项](https://github.com/AdamDHines/VPRTempo-quant#installation-and-setup) 并/或激活 conda 环境。您还需要 [Nordland](https://github.com/AdamDHines/VPRTempo-quant#nordland) 数据集，因为本教程将使用它作为示例来训练网络。

### 使用专家模块比较 VPRTempo 的结果

让我们从使用 1000 个地点训练基础模型开始，这比默认设置多 500 个。我们需要解析 `--train_new_model`、`--num_places`，以及另一个我们尚未见过的参数 `--max_module`。

`--max_module` 告诉网络每个专家模块应该学习多少地点，默认设置为 `500`。所以如果我们要训练一个新的单一网络学习 1000 个地点，我们需要将 `max_module` 增加到 1000。

In [ ]:
# Change the working directory to the main folder from tutorials
import os
os.chdir('../')

In [ ]:
# Train a single network with 1000 places
!python main.py --num_places 1000 --max_module 1000 --train_new_model

现在让我们看看它的表现。

In [ ]:
# Run the inferencing network on the singular 1000 place trained model
!python main.py --num_places 1000 --max_module 1000

这里的性能仍然相当不错，但让我们看看是否可以通过将网络拆分为模块来改进它！

现在将我们的 1000 地点网络拆分为 2 个网络，我们可以移除 `--max_module` 参数，因为默认设置为 500。相反，我们将解析 `--num_modules` 来告诉网络将事物拆分为两个模型。

In [ ]:
# Train a new 1000 place model with 2 modules
!python main.py --num_places 1000 --num_modules 2 --train_new_model

现在让我们看看它与单一模型的比较。

In [ ]:
# Run the network with 2 modules
!python main.py --num_places 1000 --num_modules 2

性能有适度的提升，但您可以想象这如何扩展到更大的网络——尤其是在考虑训练时间时。因为输出层是 one-hot 编码的，您需要随着要学习的每个地点增加输出神经元的数量。将网络拆分对 VPRTempo 有关键好处，可以减少整体训练时间，而对推理速度影响很小。

（可选）为 2500 个地点运行单一网络，或为 500 个地点运行 5 个专家模块（减少维度以加快速度）。

In [ ]:
# Optional: run a 2500 place comparison for singular vs modular networks
# Train networks
!python main.py --num_places 2500 --max_module 2500 --dims 28,28 --patches 7 --train_new_model
!python main.py --num_places 2500 --num_modules 5 --dims 28,28 --patches 7 --train_new_model
# Run inference
!python main.py --num_places 2500 --max_module 2500 --dims 28,28 --patches 7
!python main.py --num_places 2500 --num_modules 5 --dims 28,28 --patches 7

与其他教程一样，解析 `--quantize` 参数将运行完全相同的内容，但针对 VPRTempoQuant。让我们做一个快速比较。

In [ ]:
# Train networks
#!python main.py --num_places 1500 --max_module 1500 --dims 28,28 --patches 7 --train_new_model --quantize
#!python main.py --num_places 1500 --num_modules 3 --dims 28,28 --patches 7 --train_new_model --quantize
# Run inference
!python main.py --num_places 1500 --max_module 1500 --dims 28,28 --patches 7 --quantize
!python main.py --num_places 1500 --num_modules 3 --dims 28,28 --patches 7 --quantize

再一次，我们可以看到虽然准确性结果有适度的提升，但明显的改进是训练速度。因为每个网络更小，在 CPU 上拆分网络时操作的计算负担要小得多。

### 结论

本教程提供了如何使用专家模块训练网络模型的简单概述。

如果您想更深入了解，请查看一些[其他教程](https://github.com/AdamDHines/VPRTempo-quant/tree/main/tutorials)。